# Analisi della distanza dalla FDT ($I_t$) al variare di $h$

Confronto tra $h = 10^{-6}, 10^{-7}, 10^{-8}, 10^{-9}$ per il modello di Wilson-Cowan sbilanciato (Nandi et al., *Phys. Rev. E* **107**, 064307 (2023)).

**Nota**: questo notebook e' un estratto di `confronto_h_PULITO.ipynb`, ridotto alla sola analisi della violazione della fluttuazione-dissipazione ($I_t$) in funzione di $h$. Non contiene i confronti di $E_0, I_0, \Sigma_0, \Delta_0$, i grafici del tasso di entropia, ne' l'analisi di $\beta_{eff}$ e della temperatura effettiva.

**Nota sui dati**: tutte le quantita' (`E0`, `I0`, `A_EE`, `A_EI`, `A_IE`, `A_II`, `Sigma0`, `Delta0`, gli elementi della matrice ruotata `A_x,A_y,A_z,A_w`, la covarianza `sigma11,sigma12`, gli autovalori) vengono caricate direttamente dai file `.npz` prodotti dallo script di simulazione e arrotondate a `DECIMALI = 9` cifre decimali al momento del caricamento. Nessuna di queste quantita' viene ricalcolata in questo notebook: si usano sempre i valori gia' calcolati e salvati dalla simulazione (Eq. A16-A19, A24, A32-A34 del paper).

## 1. Import e parametri

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# FILE E PARAMETRI DI CONFRONTO
# ============================================================
# Valori di h da confrontare: 1e-6, 1e-7, 1e-8, 1e-9
# I nomi dei file seguono la convenzione dello script di simulazione
# originale: tag = f"h{h:.0e}"  ->  "h1e-06", "h1e-07", "h1e-08", "h1e-09"

files = {
    r"10^{-6}": "grid_dynamics_7030_h1e-06_sigmaFIX.npz",
    r"10^{-7}": "grid_dynamics_7030_h1e-07_sigmaFIX.npz",
    r"10^{-8}": "grid_dynamics_7030_h1e-08_sigmaFIX.npz",
    r"10^{-9}": "grid_dynamics_7030_h1e-09_sigmaFIX.npz",
}

# Parametri fissi del modello 
alpha = 0.1
wEE = 6.95
wII = 6.85

chiE = 0.7
chiI = 0.3

# Numero di cifre decimali a cui arrotondare i dati al caricamento
DECIMALI = 9

# Colori fissi per h, riusati in tutto il notebook per i confronti tra h
COLORS_H = {
    r"10^{-6}": "crimson",
    r"10^{-7}": "darkorange",
    r"10^{-8}": "seagreen",
    r"10^{-9}": "navy",
}


## 2. Caricamento dei dati

Un unico caricamento per tutti i campi necessari (sia per i plot di base sia per l'analisi FDT e $\beta_{eff}$): `E0, I0, Sigma0, Delta0`, gli elementi della Jacobiana (`A_EE, A_EI, A_IE, A_II`), gli elementi della matrice ruotata (`A_x, A_y, A_z, A_w`), la covarianza (`sigma11, sigma12`) e gli autovalori (`lambda1, lambda2, lambda_re, lambda_im, is_complex`).



In [2]:
# ============================================================
# CARICAMENTO DATI 
# ============================================================

data_all = {}

for h_label, filename in files.items():
    raw = np.load(filename)

    data_all[h_label] = {
        "delta_EI":   raw["delta_EI"],
        "delta_IE":   raw["delta_IE"],
        "E0":         np.round(raw["E0"], DECIMALI),
        "I0":         np.round(raw["I0"], DECIMALI),
        "Sigma0":     np.round(raw["Sigma0"], DECIMALI),
        "Delta0":     np.round(raw["Delta0"], DECIMALI),
        "A_EE":       np.round(raw["A_EE"], DECIMALI),
        "A_EI":       np.round(raw["A_EI"], DECIMALI),
        "A_IE":       np.round(raw["A_IE"], DECIMALI),
        "A_II":       np.round(raw["A_II"], DECIMALI),
        "A_x":        np.round(raw["A_x"], DECIMALI),
        "A_y":        np.round(raw["A_y"], DECIMALI),
        "A_z":        np.round(raw["A_z"], DECIMALI),
        "A_w":        np.round(raw["A_w"], DECIMALI),
        "sigma11":    np.round(raw["sigma11"], DECIMALI),
        "sigma12":    np.round(raw["sigma12"], DECIMALI),
        "lambda1":    np.round(raw["lambda1"], DECIMALI),
        "lambda2":    np.round(raw["lambda2"], DECIMALI),
        "lambda_re":  np.round(raw["lambda_re"], DECIMALI),
        "lambda_im":  np.round(raw["lambda_im"], DECIMALI),
        "is_complex": raw["is_complex"],   # flag 0/1, non arrotondato
    }

print("File caricati:")
for h_label, filename in files.items():
    print(f"  h = {h_label:>8s}  <-  {filename}")


File caricati:
  h =  10^{-6}  <-  grid_dynamics_7030_h1e-06_sigmaFIX.npz
  h =  10^{-7}  <-  grid_dynamics_7030_h1e-07_sigmaFIX.npz
  h =  10^{-8}  <-  grid_dynamics_7030_h1e-08_sigmaFIX.npz
  h =  10^{-9}  <-  grid_dynamics_7030_h1e-09_sigmaFIX.npz


## 3. Tasso di produzione di entropia

Necessario come ingrediente per il calcolo di $I_t$ (Sez. 4): non viene mostrato alcun grafico di $W$ in questo notebook, solo il calcolo della matrice `W_matrices` usata piu' avanti.

### Espressione analitica

Il tasso medio di produzione di entropia $\langle W_t\rangle/t$ e' calcolato, punto per
punto nella griglia $(\delta_{IE},\delta_{EI})$, dalla seguente espressione:

$$
\frac{\langle W_t\rangle}{t} \;=\; -\,
\frac{\left[\sqrt{\dfrac{\chi_E}{\chi_I}}\,A_{EI}\,I_0 \;-\; \sqrt{\dfrac{\chi_I}{\chi_E}}\,A_{IE}\,E_0\right]^2}
{\left(A_{EE}+A_{II}\right)\,E_0\,I_0}
$$

dove $A_{EE}, A_{EI}, A_{IE}, A_{II}$ sono gli elementi della matrice Jacobiana (Eq. A16-A19
del paper), $\chi_E=0.7,\chi_I=0.3$. Il segno meno garantisce $\langle W_t\rangle/t\ge0$
(secondo principio), perche' il denominatore $\mathrm{tr}(A)<0$ in un punto fisso stabile
mentre il numeratore (un quadrato) e' sempre $\ge0$.

In [3]:
# ============================================================
# TASSO DI PRODUZIONE DI ENTROPIA
# ============================================================

def compute_entropy_rate(data):
    """
    Calcola il tasso medio di entropia <W_t>/t (matrice 2D, stessa forma
    di E0) a partire dai campi gia' caricati per un dato h.
    """
    E0 = data["E0"]; I0 = data["I0"]
    A_EE = data["A_EE"]; A_EI = data["A_EI"]; A_IE = data["A_IE"]; A_II = data["A_II"]

    delta_term = (
        np.sqrt(chiE / chiI) * A_EI * I0
        - np.sqrt(chiI / chiE) * A_IE * E0
    )
    trA = A_EE + A_II

    numeratore = delta_term**2
    denominatore = trA * E0 * I0

    with np.errstate(divide="ignore", invalid="ignore"):
        W = -numeratore / denominatore

    return W


W_matrices = {}
for h_label, data in data_all.items():
    W_matrices[h_label] = compute_entropy_rate(data)

print("Tasso di entropia calcolato per:", list(W_matrices.keys()))


Tasso di entropia calcolato per: ['10^{-6}', '10^{-7}', '10^{-8}', '10^{-9}']


In [4]:
# ============================================================
# LINEE DI GRIGLIA USATE PER L'ANALISI DI I_t (Sez. 4)
# ============================================================
# Questi due elenchi (originariamente definiti insieme ai grafici del
# tasso di entropia lungo linee, qui rimossi) vengono riutilizzati sia
# per il calcolo di T (Sez. 4.1) sia per le linee di I_t (Sez. 4.3-4.4).

valori_dei = [-5, -2.5, -0.5, 0, 2.5, 5]   # linee orizzontali: delta_EI fissato
valori_die = [-5, -2.5, -0.5, 0, 2.5, 5]   # linee verticali: delta_IE fissato


---
# 4. Distanza dalla FDT: $I_t$

Quantifica la violazione della fluctuation-dissipation theorem (FDT) lungo linee a $\delta_{EI}$ fissato ("linee orizzontali") e a $\delta_{IE}$ fissato ("linee verticali"), per i 4 valori di $h$.

**Definizioni**:
$$x(t) = \frac{C_{\Sigma\Sigma}(t)}{C_{\Sigma\Sigma}(0)}, \qquad y(t) = \frac{\chi_{\Sigma\Sigma}(t)}{C_{\Sigma\Sigma}(0)}, \qquad \chi_{\Sigma\Sigma}(t) = \int_0^t R_{\Sigma\Sigma}(t')\,dt'$$

Se la FDT e' soddisfatta ci si aspetta $y(t)/y_\infty = 1 - x(t)$. La distanza dalla FDT e' quindi
$$I_t = \frac{1}{T}\int_0^T \left| \frac{y(t)}{y_\infty} - (1-x(t)) \right| dt$$

$C_{\Sigma\Sigma}(t)$ e $\chi_{\Sigma\Sigma}(t)$ sono calcolate analiticamente a partire da $x,y,z,w$ (Eq. A24), $\sigma_{11},\sigma_{12}$ (Eq. A32-A34) e dagli autovalori (Eq. A38-A39) — nessun valore viene ricalcolato, si usano solo le quantita' gia' salvate dalla simulazione. $y_\infty$ e' stimato come la media di $y(t)$ sull'ultimo 10% della traiettoria (`tail_mean`), piu' robusta al rumore residuo del solo ultimo punto $y(t_{max})$.


## 4.1 Come viene scelto $T$ (tempo di integrazione, UNICO per tutti gli $h$)

$T$ **non** e' un valore fisso scelto a mano, ma nemmeno diverso per ciascun $h$: e' un **unico** numero, calcolato una sola volta e usato per integrare tutte le linee, per tutti gli $h$. Cosi' $I_t$ resta sempre confrontabile direttamente tra $h$ diversi (stesso denominatore $T$ nella definizione sopra). Il calcolo, passo per passo:

1. **Tempo di rilassamento in ogni punto usato.** Per ogni punto $(\delta_{IE},\delta_{EI})$ che sta su una delle linee che uso (`valori_dei`/`valori_die`, non tutta la griglia 2D), si calcola $\tau_{slow}=-1/\lambda$ usando l'autovalore piu' lento (quello meno negativo): $\lambda_{re}$ se il regime e' complesso, $\max(\lambda_1,\lambda_2)$ se e' reale.
2. **Il caso peggiore, su TUTTI gli $h$.** Si prende il massimo di $\tau_{slow}$ su tutte le linee usate, per ciascun $h$ separatamente, e poi il massimo **tra i 4 valori** cosi' ottenuti: $\tau_{slow}^{max}$. Usare il peggiore tra tutti gli $h$ (non solo il proprio) garantisce che $T$ vada bene anche per l'$h$ piu' lento a rilassarsi.
3. **Da $\tau$ a $T$.** $T^{ideale} = K\cdot\tau_{slow}^{max}$, con $K=21$ (margine ancorato alla precisione reale dei dati, arrotondati a 9 cifre decimali: $e^{-21}\approx7.6\times10^{-10}$, coerente con la soglia di rumore $\sim10^{-9}$ introdotta dall'arrotondamento a 9 decimali di E0, I0 e delle quantita' derivate. Un $K$ maggiore, come il precedente $K=30$ ($e^{-30}\approx10^{-13}$), punterebbe a un decadimento piu' fine di quanto i dati possano risolvere).
4. **Tetto e pavimento.** $T=\mathrm{clip}(T^{ideale},\,1000,\,250000)$: il tetto evita che, vicino alla criticita' pura (dove $\tau_{slow}$ diverge per $h\to0$), il calcolo diventi troppo lungo; se il tetto viene raggiunto, viene segnalato esplicitamente come **CAPPATO**. Con $K=21$, il tetto a 250000 e' sufficiente a coprire il caso peggiore osservato (h=$10^{-9}$, $\tau_{slow}^{max}\approx11332$) senza attivare il cap.
5. **Griglia temporale.** `t_common = linspace(0, T, 200000)` — questo stesso array, identico, viene passato per tutte le linee e tutti gli $h$.


In [5]:
# ============================================================
# T UNICO PER TUTTI GLI h (l'unico tempo di integrazione usato in questa sezione)
# ============================================================

tail_fraction = 0.10
tol_x_decay = 1e-5
tol_y_plateau = 1e-5

K_TAU = 21              # quante costanti di tempo lente includere (Sez. 6.1: ancorato a ~1e-9, precisione a 9 decimali dei dati)
N_POINTS_T = 200000
T_CAP = 250000          # tetto di sicurezza sul tempo di integrazione
T_FLOOR = 1000.0        # non scendere sotto questo valore


def tau_slow_max_su_linee_usate(data, valori_dei, valori_die):
    """Tau_slow piu' grande, limitato alle righe (delta_EI fissato, in
    valori_dei) e alle colonne (delta_IE fissato, in valori_die)
    effettivamente campionate - non a tutta la griglia 2D. Usa lambda_re
    (caso complesso) o l'autovalore meno negativo tra lambda1, lambda2
    (caso reale), ignorando punti con autovalori positivi o non finiti.
    """
    delta_EI = data["delta_EI"]; delta_IE = data["delta_IE"]
    is_cplx = data["is_complex"]; lam_re = data["lambda_re"]
    L1 = data["lambda1"]; L2 = data["lambda2"]

    lam_slow = np.where(is_cplx == 1, lam_re, np.maximum(L1, L2))
    lam_slow = np.where(np.isfinite(lam_slow) & (lam_slow < 0), lam_slow, np.nan)
    tau_full = -1.0 / lam_slow

    mask = np.zeros_like(tau_full, dtype=bool)
    for target in valori_dei:
        i = int(np.argmin(np.abs(delta_EI - target)))
        mask[i, :] = True
    for target in valori_die:
        j = int(np.argmin(np.abs(delta_IE - target)))
        mask[:, j] = True

    tau_masked = np.where(mask, tau_full, np.nan)
    if np.all(np.isnan(tau_masked)):
        return np.nan, (np.nan, np.nan)

    k_flat = int(np.nanargmax(tau_masked))
    i_max, j_max = np.unravel_index(k_flat, tau_masked.shape)
    return float(np.nanmax(tau_masked)), (delta_EI[i_max], delta_IE[j_max])


tau_max_per_h = {}
print(f"{'h':>10s} | {'tau_slow max':>13s} | {'(delta_EI, delta_IE)':>22s}")
for h_label, data in data_all.items():
    tau_max, loc = tau_slow_max_su_linee_usate(data, valori_dei, valori_die)
    tau_max_per_h[h_label] = tau_max
    print(f"{h_label:>10s} | {tau_max:13.3f} | ({loc[0]:>8.3f},{loc[1]:>8.3f})")

tau_slow_max_globale = max(tau_max_per_h.values())
T_max = float(np.clip(K_TAU * tau_slow_max_globale, T_FLOOR, T_CAP))
flag_cap = "  <-- CAPPATO" if K_TAU * tau_slow_max_globale > T_CAP else ""

print(f"\ntau_slow_max su tutti gli h = {tau_slow_max_globale:.3f}")
print(f"T (unico, usato per TUTTI gli h) = {T_max:.0f}{flag_cap}")

t_common = np.linspace(0, T_max, N_POINTS_T)


         h |  tau_slow max |   (delta_EI, delta_IE)
   10^{-6} |      1581.078 | (   0.000,   0.000)
   10^{-7} |      4878.025 | (   0.000,   0.000)
   10^{-8} |      9800.847 | (   0.000,   0.000)
   10^{-9} |     11331.958 | (   0.000,   0.000)

tau_slow_max su tutti gli h = 11331.958
T (unico, usato per TUTTI gli h) = 237971


## 4.2 Funzioni core

In [6]:
import numpy as np


def _trapz(y, x):
    """Wrapper compatibile sia con numpy < 2.0 (np.trapz) sia >= 2.0 (np.trapezoid)."""
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)


def compute_I_t_along_line(data, W_ent, axis, target, t,
                            tail_fraction=0.10, tol_x_decay=1e-5, tol_y_plateau=1e-5,
                            yinf_method="tail_mean", escludi_se_non_decay=False):
    """
    Calcola I_t lungo una linea (delta_EI fissato se axis="dei", delta_IE
    fissato se axis="die"), integrando su t (SEMPRE lo stesso array t_common
    per tutti i punti e tutti gli h - vedi Sez. 6.1).
    """
    delta_EI_vals = data["delta_EI"]; delta_IE_vals = data["delta_IE"]

    if axis == "dei":
        i = int(np.argmin(np.abs(delta_EI_vals - target)))
        var_vals = delta_IE_vals
        sl = lambda arr: arr[i, :]
    elif axis == "die":
        j = int(np.argmin(np.abs(delta_IE_vals - target)))
        var_vals = delta_EI_vals
        sl = lambda arr: arr[:, j]
    else:
        raise ValueError("axis deve essere 'dei' o 'die'")

    X = sl(data["A_x"]); Y = sl(data["A_y"]); Z = sl(data["A_z"]); Wm = sl(data["A_w"])
    s11 = sl(data["sigma11"]); s12 = sl(data["sigma12"])
    is_cplx = sl(data["is_complex"])
    a_ = sl(data["lambda_re"]); b_ = sl(data["lambda_im"])
    L1 = sl(data["lambda1"]); L2 = sl(data["lambda2"])
    Wt = sl(W_ent)

    n = len(var_vals)
    Ntail = max(2, round(tail_fraction * len(t)))

    I_line = np.full(n, np.nan)
    pass_line = np.full(n, np.nan)
    maxx_line = np.full(n, np.nan)
    rely_line = np.full(n, np.nan)
    reason = np.full(n, "", dtype=object)

    for k in range(n):

        if not np.isfinite(Wt[k]) or Wt[k] == 0:
            reason[k] = "entropia_non_valida"
            continue

        if is_cplx[k] == 1:
            an, bn = a_[k], b_[k]
            if bn == 0 or not np.isfinite(bn):
                reason[k] = "denominatore_degenere"
                continue
            expat = np.exp(an * t); costbt = np.cos(bn * t); sintbt = np.sin(bn * t)
            C_sig = (s11[k]*expat*costbt
                     + (((X[k]-an)*s11[k]+Y[k]*s12[k])/bn)*expat*sintbt)
            C0 = s11[k]
            denom = bn*(an**2+bn**2)
            if abs(denom) < 1e-14:
                denom = denom + np.sign(denom if denom != 0 else 1.0)*1e-14
            integrale = -(bn*(X[k]-2*an)*(costbt*expat-1)
                          + sintbt*expat*(an**2-bn**2-X[k]*an)) / denom
        elif is_cplx[k] == 0:
            L1k, L2k = L1[k], L2[k]
            with np.errstate(invalid="ignore"):
                rad = np.sqrt((X[k]-Wm[k])**2 + 4*Y[k]*Z[k])
            if (not np.isfinite(rad)) or rad == 0:
                rad = 1e-14
            C_sig = (((X[k]-L2k)*s11[k]+Y[k]*s12[k])*np.exp(L1k*t)
                      - ((X[k]-L1k)*s11[k]+Y[k]*s12[k])*np.exp(L2k*t)) / rad
            C0 = (((X[k]-L2k)*s11[k]+Y[k]*s12[k]) - ((X[k]-L1k)*s11[k]+Y[k]*s12[k])) / rad
            if abs(C0) < 1e-14:
                reason[k] = "denominatore_degenere"
                continue
            denom = L1k*L2k*rad
            if abs(denom) < 1e-14:
                denom = denom + np.sign(denom if denom != 0 else 1.0)*1e-14
            integrale = (L2k*(X[k]-L2k)*(np.exp(L1k*t)-1)
                         - L1k*(X[k]-L1k)*(np.exp(L2k*t)-1)) / denom
        else:
            reason[k] = "campo_non_valido"
            continue

        if abs(C0) < 1e-14:
            reason[k] = "denominatore_degenere"
            continue

        x_vals = C_sig / C0
        y_vals = integrale / C0

        x_tail = x_vals[-Ntail:]; y_tail = y_vals[-Ntail:]
        max_x_tail = np.max(np.abs(x_tail))
        y_inf = y_vals[-1] if yinf_method == "end" else np.mean(y_tail)

        max_y_tail = np.max(np.abs(y_tail - y_inf))
        rel_y_tail = max_y_tail / max(abs(y_inf), 1e-14)
        pass_decay = (max_x_tail < tol_x_decay) and (rel_y_tail < tol_y_plateau)

        maxx_line[k] = max_x_tail; rely_line[k] = rel_y_tail; pass_line[k] = float(pass_decay)

        if not pass_decay and escludi_se_non_decay:
            reason[k] = "non_decay_escluso"
            continue

        if abs(y_inf) < 1e-14 or not np.isfinite(y_inf):
            reason[k] = "y_inf_degenere"
            continue

        f_num = y_vals / y_inf
        f_FDT = 1 - x_vals
        delta_t = f_num - f_FDT
        mask_t = np.isfinite(delta_t) & np.isfinite(t)
        if np.sum(mask_t) < 2:
            reason[k] = "maschera_insufficiente"
            continue

        t_int = t[mask_t]; delta_int = delta_t[mask_t]
        I_line[k] = _trapz(np.abs(delta_int), t_int) / (t_int[-1]-t_int[0])
        reason[k] = "ok"

    return {"var_vals": var_vals, "I": I_line, "pass": pass_line,
            "max_x_tail": maxx_line, "rel_y_tail": rely_line,
            "W": Wt, "reason": reason}


def run_fdt_lines(data_all, W_matrices, axis, target_values, t,
                   tail_fraction=0.10, tol_x_decay=1e-5, tol_y_plateau=1e-5,
                   yinf_method="tail_mean", escludi_se_non_decay=False):
    """Esegue compute_I_t_along_line per tutte le linee e tutti gli h."""
    results = {}
    for h_label, data in data_all.items():
        W_ent = W_matrices[h_label]
        results[h_label] = {}
        for target in target_values:
            results[h_label][target] = compute_I_t_along_line(
                data, W_ent, axis, target, t,
                tail_fraction=tail_fraction, tol_x_decay=tol_x_decay,
                tol_y_plateau=tol_y_plateau, yinf_method=yinf_method,
                escludi_se_non_decay=escludi_se_non_decay,
            )
    return results


PLOTLY_COLORS = [
    "rgb(31,119,180)", "rgb(174,199,232)", "rgb(214,39,40)",
    "rgb(44,160,44)", "rgb(255,127,14)", "rgb(148,103,189)",
]
PLOTLY_SYMBOLS = ["circle", "square", "triangle-up", "diamond", "pentagon", "star"]


def plot_I_t_lines_interactive(results, target_values, xlabel, legend_prefix, title, yscale="linear"):
    """Griglia 2x2 (una per h), tooltip al passaggio del mouse/tocco."""
    h_labels = list(results.keys())
    n_cols = 2
    n_rows = int(np.ceil(len(h_labels) / n_cols))   # dinamico: funziona con qualsiasi numero di h

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[f"h={h_label}" for h_label in h_labels],
        horizontal_spacing=0.08, vertical_spacing=0.12,
    )

    for idx, h_label in enumerate(h_labels):
        row, col = idx // n_cols + 1, idx % n_cols + 1
        per_target = results[h_label]

        for j, target in enumerate(target_values):
            res = per_target[target]
            var_vals = res["var_vals"]
            I_vals = res["I"].astype(float).copy()

            order = np.argsort(var_vals)
            var_sorted = var_vals[order]
            I_sorted = I_vals[order]
            if yscale == "log":
                I_sorted = np.where(I_sorted > 0, I_sorted, np.nan)

            color = PLOTLY_COLORS[j % len(PLOTLY_COLORS)]
            symbol = PLOTLY_SYMBOLS[j % len(PLOTLY_SYMBOLS)]

            fig.add_trace(
                go.Scatter(
                    x=var_sorted, y=I_sorted, mode="lines+markers",
                    name=f"{legend_prefix} = {target:g}",
                    legendgroup=f"target_{target}", showlegend=(idx == 0),
                    line=dict(color=color, dash="dash", width=1.5),
                    marker=dict(symbol=symbol, size=7, color=color),
                    hovertemplate=(f"{legend_prefix} = {target:g}<br>" + xlabel +
                                   " = %{x:.4g}<br>I_t = %{y:.6g}<extra></extra>"),
                ),
                row=row, col=col,
            )

        fig.update_xaxes(title_text=xlabel, row=row, col=col)
        fig.update_yaxes(title_text="I_t", type="log" if yscale == "log" else "linear", row=row, col=col)

    fig.update_layout(title=title, height=400 * n_rows, width=1100, hovermode="closest",
                       legend=dict(title=legend_prefix))
    return fig


## 4.3 Linee a $\delta_{EI}$ fissato (orizzontali)

In [7]:
results_dei = run_fdt_lines(
    data_all, W_matrices,
    axis="dei", target_values=valori_dei, t=t_common,
    tail_fraction=tail_fraction, tol_x_decay=tol_x_decay, tol_y_plateau=tol_y_plateau,
    yinf_method="tail_mean", escludi_se_non_decay=False,
)
print("Calcolo completato (linee a delta_EI fissato).")


Calcolo completato (linee a delta_EI fissato).


In [8]:
fig_dei_log = plot_I_t_lines_interactive(
    results_dei, valori_dei, "delta_IE", "delta_EI",
    "Distanza FDT I_t (scala log), delta_EI fissato",
    yscale="log",
)
fig_dei_log.show()


## 4.4 Linee a $\delta_{IE}$ fissato (verticali)

In [9]:
results_die = run_fdt_lines(
    data_all, W_matrices,
    axis="die", target_values=valori_die, t=t_common,
    tail_fraction=tail_fraction, tol_x_decay=tol_x_decay, tol_y_plateau=tol_y_plateau,
    yinf_method="tail_mean", escludi_se_non_decay=False,
)
print("Calcolo completato (linee a delta_IE fissato).")


Calcolo completato (linee a delta_IE fissato).


In [10]:
fig_die_log = plot_I_t_lines_interactive(
    results_die, valori_die, "delta_EI", "delta_IE",
    "Distanza FDT I_t (scala log), delta_IE fissato",
    yscale="log",
)
fig_die_log.show()


## 4.5 Diagnostica dei picchi (linee a $\delta_{EI}$ fissato)

In [11]:
import re
from pathlib import Path

def h_label_to_float(h_label):
    """Converte etichette tipo r"10^{-6}" in 1e-6."""
    s = str(h_label)
    m = re.search(r"10\^\{(-?\d+)\}", s)
    if m:
        return 10.0 ** int(m.group(1))
    try:
        return float(s)
    except Exception:
        return np.nan


def _beta_eff_closed_form(data, W_ent, i, j):
    """Stima beta_eff = y(infinito) con la formula chiusa (Sez. 7)."""
    X = data["A_x"][i, j]; s11 = data["sigma11"][i, j]; is_cplx = data["is_complex"][i, j]
    if not np.isfinite(W_ent[i, j]) or W_ent[i, j] == 0:
        return np.nan
    if is_cplx == 1:
        a = data["lambda_re"][i, j]; b = data["lambda_im"][i, j]
        denom = s11 * (a**2 + b**2)
        if abs(denom) < 1e-14 or not np.isfinite(denom):
            return np.nan
        return (X - 2*a) / denom
    elif is_cplx == 0:
        L1 = data["lambda1"][i, j]; L2 = data["lambda2"][i, j]
        denom = L1 * L2 * (L1 - L2) * s11
        if abs(denom) < 1e-14 or not np.isfinite(denom):
            return np.nan
        return (L1*(X-L1) - L2*(X-L2)) / denom
    return np.nan


def build_peak_diagnostics(results, data_all, W_matrices, axis="dei", target_values=None):
    """Tabella dei massimi di I_t per ogni h e ogni linea."""
    if target_values is None:
        target_values = valori_dei if axis == "dei" else valori_die

    rows = []
    for h_label, per_target in results.items():
        data = data_all[h_label]; W_ent = W_matrices[h_label]
        h_float = h_label_to_float(h_label)

        for target in target_values:
            res = per_target[target]
            var_vals = np.asarray(res["var_vals"]); I_vals = np.asarray(res["I"])
            valid = np.isfinite(I_vals) & (I_vals > 0)

            if np.sum(valid) < 3:
                rows.append({"h_label": h_label, "h": h_float, "target": target,
                             "n_valid": int(np.sum(valid)), "deltaIE_peak": np.nan,
                             "deltaEI_peak": np.nan, "Imax": np.nan,
                             "reason": "meno_di_3_punti_validi"})
                continue

            idx_valid = np.where(valid)[0]
            var_valid = var_vals[valid]; I_valid = I_vals[valid]
            order = np.argsort(var_valid)
            idx_valid = idx_valid[order]

            imax_local = int(np.nanargmax(I_valid[order]))
            k_peak = int(idx_valid[imax_local])
            Imax = float(I_valid[order][imax_local])

            if axis == "dei":
                i = int(np.argmin(np.abs(data["delta_EI"] - target))); j = k_peak
            else:
                j = int(np.argmin(np.abs(data["delta_IE"] - target))); i = k_peak

            delta_EI_peak = float(data["delta_EI"][i]); delta_IE_peak = float(data["delta_IE"][j])
            E0 = float(data["E0"][i, j]); I0 = float(data["I0"][i, j])

            is_cplx = int(data["is_complex"][i, j])
            if is_cplx == 1:
                lambda_slow = float(data["lambda_re"][i, j])
            else:
                lambda_slow = float(max(data["lambda1"][i, j], data["lambda2"][i, j]))
            tau_slow = float(-1/lambda_slow) if np.isfinite(lambda_slow) and lambda_slow < 0 else np.nan

            beta_eff = _beta_eff_closed_form(data, W_ent, i, j)
            pass_peak = float(res["pass"][k_peak]) if "pass" in res else np.nan
            reason_peak = str(res["reason"][k_peak]) if "reason" in res else ""

            rows.append({
                "h_label": h_label, "h": h_float, "target": target,
                "deltaEI_peak": delta_EI_peak, "deltaIE_peak": delta_IE_peak,
                "Imax": Imax, "beta_eff_formula": beta_eff,
                "lambda_slow": lambda_slow, "tau_slow": tau_slow,
                "E0": E0, "I0": I0, "campo_is_complex": is_cplx,
                "pass_decay": pass_peak, "reason": reason_peak,
            })

    return pd.DataFrame(rows)


T_peaks_dei = build_peak_diagnostics(results_dei, data_all, W_matrices, axis="dei", target_values=valori_dei)
print("Tabella dei massimi di I_t lungo le curve a delta_EI fissato:")
display(T_peaks_dei)


Tabella dei massimi di I_t lungo le curve a delta_EI fissato:


,h_label,h,target,deltaEI_peak,deltaIE_peak,Imax,beta_eff_formula,lambda_slow,tau_slow,E0,I0,campo_is_complex,pass_decay,reason
0,10^{-6},1.000000e-06,-5.0,-5.0,-1.2,4.924072e-07,14.157846,-1.099832,0.909230,0.909083,0.723907,0,1.0,ok
1,10^{-6},1.000000e-06,-2.5,-2.5,-1.0,5.038779e-07,14.348459,-1.095578,0.912760,0.908737,0.745329,0,1.0,ok
2,10^{-6},1.000000e-06,-0.5,-0.5,0.2,4.757281e-05,0.695768,-0.841343,1.188575,0.865191,0.827066,1,1.0,ok
3,10^{-6},1.000000e-06,0.0,0.0,-0.4,4.490694e-04,0.055089,-0.948217,1.054611,0.856422,0.768534,1,1.0,ok
4,10^{-6},1.000000e-06,2.5,2.5,-2.2,1.534062e-05,-0.955403,-0.707388,1.413651,0.836699,0.561404,0,1.0,ok
5,10^{-6},1.000000e-06,5.0,5.0,-3.3,9.982182e-06,1.210098,-0.735423,1.359762,0.855751,0.444288,0,1.0,ok
6,10^{-7},1.000000e-07,-5.0,-5.0,-1.2,4.924072e-07,14.157844,-1.099832,0.909230,0.909083,0.723907,0,1.0,ok
7,10^{-7},1.000000e-07,-2.5,-2.5,-1.0,5.038780e-07,14.348458,-1.095578,0.912760,0.908737,0.745329,0,1.0,ok
8,10^{-7},1.000000e-07,-0.5,-0.5,0.2,4.757347e-05,0.695758,-0.841343,1.188575,0.865191,0.827065,1,1.0,ok
9,10^{-7},1.000000e-07,0.0,0.0,-0.4,4.490925e-04,0.055086,-0.948217,1.054611,0.856422,0.768534,1,1.0,ok


### Perche' esistono i picchi a $\delta_{EI}=2.5$ e $5$ (e perche' NON dipendono da $h$)

A differenza di un eventuale picco vicino alla criticita', qui `pass_decay=1`, `tau_slow~O(1)` (rilassamento velocissimo): non e' un problema di tempo di integrazione ne' di vero rallentamento critico.

**Ipotesi verificabile analiticamente.** Nel caso reale, la forma chiusa di $\beta_{eff}=y_\infty$ (Sez. 7) si fattorizza come
$$\beta_{eff} = \frac{(L_1-L_2)\,(X - \mathrm{tr}A)}{L_1 L_2 (L_1-L_2)\,\sigma_{11}} \;\propto\; X - (x+w)= -w,$$
cioe' **$\beta_{eff}=0$ esattamente quando $w=0$** (l'elemento $w$ della matrice $A$ in variabili $\Sigma,\Delta$, Eq. A24), indipendentemente da $L_1,L_2$ singolarmente. Poiche' $I_t$ e' normalizzato dividendo per $y_\infty=\beta_{eff}$, nel punto dove $w$ cambia segno $I_t$ diverge **per costruzione** (singolarita' di normalizzazione, non rumore ne' troncamento).

Poiche' lo zero di $w(\delta_{IE})$ a $\delta_{EI}$ fissato e' un luogo puramente deterministico (dipende da $A$, non dal rumore), altezza e posizione del picco non cambiano con $h$ lontano dalla regione critica.


In [12]:
# ============================================================
# VERIFICA: lo zero di A_w coincide con il picco di I_t? (delta_EI fissato)
# ============================================================

targets_check = [2.5, 5.0]
h_check = list(data_all.keys())[0]   # un h qualsiasi: se l'ipotesi e' giusta non cambia nulla
data_chk = data_all[h_check]

fig_w_check = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f"delta_EI={t}, h={h_check}" for t in targets_check],
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
)

for col, target in enumerate(targets_check, start=1):
    i = int(np.argmin(np.abs(data_chk["delta_EI"] - target)))
    die_line = data_chk["delta_IE"]
    w_line = data_chk["A_w"][i, :]
    order = np.argsort(die_line)

    res = results_dei[h_check][target]
    var_vals = res["var_vals"]
    I_vals = res["I"].copy()
    I_vals[(I_vals <= 0) | (~np.isfinite(I_vals))] = np.nan
    order_I = np.argsort(var_vals)

    fig_w_check.add_trace(
        go.Scatter(x=var_vals[order_I], y=I_vals[order_I], mode="markers+lines",
                   marker=dict(color="crimson", size=5), line=dict(color="crimson"),
                   name="I_t", showlegend=(col == 1),
                   hovertemplate="delta_IE=%{x:.3g}<br>I_t=%{y:.4g}<extra></extra>"),
        row=1, col=col, secondary_y=False,
    )
    fig_w_check.add_trace(
        go.Scatter(x=die_line[order], y=w_line[order], mode="lines", line=dict(color="navy"),
                   name="w (A_w)", showlegend=(col == 1),
                   hovertemplate="delta_IE=%{x:.3g}<br>w=%{y:.4g}<extra></extra>"),
        row=1, col=col, secondary_y=True,
    )
    fig_w_check.add_hline(y=0, line=dict(color="gray", dash="dash", width=1), row=1, col=col, secondary_y=True)
    fig_w_check.update_xaxes(title_text="delta_IE", row=1, col=col)
    fig_w_check.update_yaxes(title_text="I_t", type="log", row=1, col=col, secondary_y=False)
    fig_w_check.update_yaxes(title_text="w", row=1, col=col, secondary_y=True)

fig_w_check.update_layout(title="delta_EI fissato: confronto tra il picco di I_t e lo zero di w",
                          height=450, width=1150)
fig_w_check.show()

print("Se il picco rosso e l'incrocio dello zero (linea tratteggiata) della curva blu")
print("cadono nello stesso delta_IE, l'ipotesi w=0 come origine del picco e' confermata.")


Se il picco rosso e l'incrocio dello zero (linea tratteggiata) della curva blu
cadono nello stesso delta_IE, l'ipotesi w=0 come origine del picco e' confermata.


## 4.6 Diagnostica dei picchi (linee a $\delta_{IE}$ fissato) e verifica $w=0$

In [13]:
T_peaks_die = build_peak_diagnostics(results_die, data_all, W_matrices, axis="die", target_values=valori_die)
print("Tabella dei massimi di I_t lungo le curve a delta_IE fissato:")
display(T_peaks_die)


Tabella dei massimi di I_t lungo le curve a delta_IE fissato:


,h_label,h,target,deltaEI_peak,deltaIE_peak,Imax,beta_eff_formula,lambda_slow,tau_slow,E0,I0,campo_is_complex,pass_decay,reason
0,10^{-6},1.000000e-06,-5.0,7.0,-5.0,3.217453e-07,12.474234,-1.089827,0.917577,0.908479,0.253655,0,1.0,ok
1,10^{-6},1.000000e-06,-2.5,3.0,-2.5,2.946509e-05,0.471146,-0.734708,1.361086,0.848581,0.534436,0,1.0,ok
2,10^{-6},1.000000e-06,-0.5,0.1,-0.5,3.831954e-04,-0.061749,-0.969310,1.031662,0.854695,0.757584,1,1.0,ok
3,10^{-6},1.000000e-06,0.0,-0.3,0.0,3.423107e-05,-0.843513,-0.829422,1.205659,0.854913,0.803839,1,1.0,ok
4,10^{-6},1.000000e-06,2.5,-1.1,2.5,3.757911e-05,1.725346,-0.391837,2.552080,0.875389,0.906390,0,1.0,ok
5,10^{-6},1.000000e-06,5.0,-1.1,5.0,3.965943e-04,0.180771,-0.312264,3.202417,0.872421,0.909054,0,1.0,ok
6,10^{-7},1.000000e-07,-5.0,7.0,-5.0,3.217452e-07,12.474234,-1.089827,0.917577,0.908479,0.253655,0,1.0,ok
7,10^{-7},1.000000e-07,-2.5,3.0,-2.5,2.946450e-05,0.471155,-0.734708,1.361085,0.848582,0.534436,0,1.0,ok
8,10^{-7},1.000000e-07,-0.5,0.1,-0.5,3.831825e-04,-0.061751,-0.969310,1.031661,0.854695,0.757584,1,1.0,ok
9,10^{-7},1.000000e-07,0.0,-0.3,0.0,3.423079e-05,-0.843519,-0.829422,1.205659,0.854913,0.803838,1,1.0,ok


In [14]:
# ============================================================
# VERIFICA: lo zero di A_w coincide con il picco di I_t? (delta_IE fissato)
# ============================================================

targets_check_die = [-0.5, 0.0, 2.5]
h_check_die = list(data_all.keys())[0]
data_chk_die = data_all[h_check_die]

fig_w_check_die = make_subplots(
    rows=1, cols=len(targets_check_die),
    subplot_titles=[f"delta_IE={t}, h={h_check_die}" for t in targets_check_die],
    specs=[[{"secondary_y": True}] * len(targets_check_die)],
)

for col, target in enumerate(targets_check_die, start=1):
    j = int(np.argmin(np.abs(data_chk_die["delta_IE"] - target)))
    dei_line = data_chk_die["delta_EI"]
    w_line = data_chk_die["A_w"][:, j]
    order = np.argsort(dei_line)

    res = results_die[h_check_die][target]
    var_vals = res["var_vals"]
    I_vals = res["I"].copy()
    I_vals[(I_vals <= 0) | (~np.isfinite(I_vals))] = np.nan
    order_I = np.argsort(var_vals)

    fig_w_check_die.add_trace(
        go.Scatter(x=var_vals[order_I], y=I_vals[order_I], mode="markers+lines",
                   marker=dict(color="crimson", size=5), line=dict(color="crimson"),
                   name="I_t", showlegend=(col == 1),
                   hovertemplate="delta_EI=%{x:.3g}<br>I_t=%{y:.4g}<extra></extra>"),
        row=1, col=col, secondary_y=False,
    )
    fig_w_check_die.add_trace(
        go.Scatter(x=dei_line[order], y=w_line[order], mode="lines", line=dict(color="navy"),
                   name="w (A_w)", showlegend=(col == 1),
                   hovertemplate="delta_EI=%{x:.3g}<br>w=%{y:.4g}<extra></extra>"),
        row=1, col=col, secondary_y=True,
    )
    fig_w_check_die.add_hline(y=0, line=dict(color="gray", dash="dash", width=1), row=1, col=col, secondary_y=True)
    fig_w_check_die.update_xaxes(title_text="delta_EI", row=1, col=col)
    fig_w_check_die.update_yaxes(title_text="I_t", type="log", row=1, col=col, secondary_y=False)
    fig_w_check_die.update_yaxes(title_text="w", row=1, col=col, secondary_y=True)

fig_w_check_die.update_layout(title="delta_IE fissato: confronto tra il picco di I_t e lo zero di w",
                              height=450, width=1150)
fig_w_check_die.show()

print("Stesso controllo della cella precedente, ripetuto per le linee a delta_IE fissato:")
print("se il picco rosso e l'incrocio della curva blu coincidono, e' lo stesso meccanismo (w=0).")


Stesso controllo della cella precedente, ripetuto per le linee a delta_IE fissato:
se il picco rosso e l'incrocio della curva blu coincidono, e' lo stesso meccanismo (w=0).


## 4.7 Comportamento dei picchi al variare di $h$

In [15]:
# ============================================================
# COMPORTAMENTO DEI PICCHI AL VARIARE DI h (colori fissi e coerenti tra i 4 pannelli)
# ============================================================
# FIX rispetto alla versione precedente: prima i colori erano assegnati in
# automatico da Plotly in base all'ordine di aggiunta delle tracce, quindi
# lo stesso target (es. delta_EI=2.5) poteva avere un colore diverso da un
# pannello all'altro. Ora ogni target ha un colore FISSO, esplicito,
# identico nei 4 pannelli.

TARGET_COLORS = {
    -5: "rgb(31,119,180)", -2.5: "rgb(174,199,232)", -0.5: "rgb(214,39,40)",
    0: "rgb(44,160,44)", 2.5: "rgb(255,127,14)", 5: "rgb(148,103,189)",
}


def plot_peak_summary(T_peaks, targets=(2.5, 5), fixed_label="delta_EI"):
    """Altezza e tempo di rilassamento del picco, vs h.

    fixed_label: nome della variabile fissata lungo la linea (usato solo
    per l'etichetta in legenda/titolo) - "delta_EI" per T_peaks_dei,
    "delta_IE" per T_peaks_die. Non cambia il calcolo, solo il testo
    mostrato, per evitare di etichettare erroneamente un confronto a
    delta_IE fissato come se fosse "delta_EI=...".
    """
    T = T_peaks[T_peaks["target"].isin(targets)].copy().sort_values("h")

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Altezza del massimo (Imax)", "Tempo di rilassamento al picco (tau_slow)"],
    )

    for target in targets:
        sub = T[T["target"] == target].sort_values("h")
        color = TARGET_COLORS.get(target, "black")
        label = f"{fixed_label}={target}"

        fig.add_trace(
            go.Scatter(x=sub["h"], y=sub["Imax"], mode="markers+lines",
                       line=dict(color=color), marker=dict(color=color),
                       name=label, legendgroup=label, showlegend=True,
                       hovertemplate=f"{label}<br>h=%{{x:.3g}}<br>Imax=%{{y:.4g}}<extra></extra>"),
            row=1, col=1,
        )
        fig.add_trace(
            go.Scatter(x=sub["h"], y=sub["tau_slow"], mode="markers+lines",
                       line=dict(color=color), marker=dict(color=color),
                       name=label, legendgroup=label, showlegend=False,
                       hovertemplate=f"{label}<br>h=%{{x:.3g}}<br>tau_slow=%{{y:.4g}}<extra></extra>"),
            row=1, col=2,
        )

    fig.update_xaxes(title_text="h", type="log", row=1, col=1)
    fig.update_yaxes(title_text="Imax", type="log", row=1, col=1)
    fig.update_xaxes(title_text="h", type="log", row=1, col=2)
    fig.update_yaxes(title_text="tau_slow", type="log", row=1, col=2)

    fig.update_layout(height=450, width=1000,
                       title=f"Comportamento dei picchi al variare di h ({fixed_label} fissato)")
    fig.show()


plot_peak_summary(T_peaks_dei, targets=(0, 2.5, 5), fixed_label="delta_EI")
plot_peak_summary(T_peaks_die, targets=(0, 2.5, 5), fixed_label="delta_IE")


In [16]:
# ============================================================
# CONFRONTO DI Imax TRA I 4 h, PER OGNI LINEA (dei e die)
# ============================================================
# Risponde direttamente alla domanda "il massimo cambia con h?":
# una tabella con una riga per ciascun valore della linea fissata e
# una colonna per ciascun h, piu' un rapporto max/min e un verdetto.

def build_Imax_comparison_table(T_peaks, soglia_rapporto=1.10):
    """
    Pivotta la tabella dei picchi (T_peaks_dei o T_peaks_die) in modo
    da avere: una riga per target (delta_EI o delta_IE fissato), una
    colonna per ciascun h con il relativo Imax.

    soglia_rapporto: se max(Imax)/min(Imax) tra i 4 h supera questa
    soglia, il picco viene marcato come "CAMBIA" (default: 10% di
    variazione).
    """
    pivot = T_peaks.pivot_table(index="target", columns="h_label", values="Imax")

    # ordina le colonne per valore numerico di h (10^-6, 10^-7, 10^-8, 10^-9)
    pivot = pivot[sorted(pivot.columns, key=h_label_to_float)]

    pivot["min_Imax"] = pivot.min(axis=1)
    pivot["max_Imax"] = pivot.max(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        pivot["rapporto_max/min"] = pivot["max_Imax"] / pivot["min_Imax"]

    pivot["verdetto"] = np.where(
        pivot["rapporto_max/min"] > soglia_rapporto,
        "CAMBIA con h", "non cambia (~costante)"
    )

    return pivot


print(f"=== Imax per ciascuna linea a delta_EI fissato, confrontato tra i 4 h ===")
print(f"(soglia per 'CAMBIA': rapporto max/min > 1.10, cioe' oltre il 10% di variazione)\n")
T_Imax_dei = build_Imax_comparison_table(T_peaks_dei)
display(T_Imax_dei)

print(f"\n=== Imax per ciascuna linea a delta_IE fissato, confrontato tra i 4 h ===\n")
T_Imax_die = build_Imax_comparison_table(T_peaks_die)
display(T_Imax_die)

=== Imax per ciascuna linea a delta_EI fissato, confrontato tra i 4 h ===
(soglia per 'CAMBIA': rapporto max/min > 1.10, cioe' oltre il 10% di variazione)



h_label,10^{-9},10^{-8},10^{-7},10^{-6},min_Imax,max_Imax,rapporto_max/min,verdetto
target,,,,,,,,
-5.0,4.924072e-07,4.924072e-07,4.924072e-07,4.924072e-07,4.924072e-07,4.924072e-07,1.000000,non cambia (~costante)
-2.5,5.038780e-07,5.038780e-07,5.038780e-07,5.038779e-07,5.038779e-07,5.038780e-07,1.000000,non cambia (~costante)
-0.5,4.757354e-05,4.757354e-05,4.757347e-05,4.757281e-05,4.757281e-05,4.757354e-05,1.000015,non cambia (~costante)
0.0,4.490951e-04,4.490949e-04,4.490925e-04,4.490694e-04,4.490694e-04,4.490951e-04,1.000057,non cambia (~costante)
2.5,1.534076e-05,1.534076e-05,1.534074e-05,1.534062e-05,1.534062e-05,1.534076e-05,1.000009,non cambia (~costante)
5.0,9.982036e-06,9.982038e-06,9.982051e-06,9.982182e-06,9.982036e-06,9.982182e-06,1.000015,non cambia (~costante)



=== Imax per ciascuna linea a delta_IE fissato, confrontato tra i 4 h ===



h_label,10^{-9},10^{-8},10^{-7},10^{-6},min_Imax,max_Imax,rapporto_max/min,verdetto
target,,,,,,,,
-5.0,3.217452e-07,3.217452e-07,3.217452e-07,3.217453e-07,3.217452e-07,3.217453e-07,1.000000,non cambia (~costante)
-2.5,2.946444e-05,2.946445e-05,2.946450e-05,2.946509e-05,2.946444e-05,2.946509e-05,1.000022,non cambia (~costante)
-0.5,3.831809e-04,3.831812e-04,3.831825e-04,3.831954e-04,3.831809e-04,3.831954e-04,1.000038,non cambia (~costante)
0.0,3.423076e-05,3.423077e-05,3.423079e-05,3.423107e-05,3.423076e-05,3.423107e-05,1.000009,non cambia (~costante)
2.5,3.758085e-05,3.758083e-05,3.758067e-05,3.757911e-05,3.757911e-05,3.758085e-05,1.000046,non cambia (~costante)
5.0,3.968109e-04,3.968089e-04,3.967894e-04,3.965943e-04,3.965943e-04,3.968109e-04,1.000546,non cambia (~costante)


In [17]:
# ============================================================
# CONFRONTO ESPLICITO (numerico, non a occhio): il picco di I_t
# coincide con lo zero di w? per TUTTI gli h, delta_EI=2.5 e 5
# ============================================================
# NB: la griglia e' discreta (passo 0.1 in delta_IE), quindi "lo zero di w"
# nel senso stretto (w=0 esatto) in generale cade TRA due punti di griglia,
# non esattamente su uno di essi. Quello che possiamo controllare e' se il
# PUNTO DI GRIGLIA dove |w| e' minimo (il piu' vicino allo zero continuo)
# coincide con il punto di griglia dove I_t e' massimo - e quanto vale w
# li', per vedere quanto e' "piccolo" davvero (non zero esatto).

grid_step = np.mean(np.diff(np.sort(data_all[list(data_all.keys())[0]]["delta_IE"])))
tol_match = 0.51 * grid_step   # tollera al massimo un passo di griglia di differenza

rows_check = []

for h_label, data in data_all.items():
    for target in [2.5, 5.0]:

        # --- posizione del picco di I_t, dalla tabella gia' calcolata (Sez. 6.5) ---
        row_peak = T_peaks_dei[(T_peaks_dei["h_label"] == h_label) & (T_peaks_dei["target"] == target)]
        die_peak = float(row_peak["deltaIE_peak"].iloc[0])
        Imax = float(row_peak["Imax"].iloc[0])

        # --- punto di griglia dove |w| e' minimo lungo la stessa linea ---
        i = int(np.argmin(np.abs(data["delta_EI"] - target)))
        die_line = data["delta_IE"]
        w_line = data["A_w"][i, :]
        k_zero = int(np.argmin(np.abs(w_line)))
        die_wzero = float(die_line[k_zero])
        w_al_minimo = float(w_line[k_zero])   # <-- quanto vale w LI', non zero esatto

        match = bool(np.isclose(die_peak, die_wzero, atol=tol_match))

        rows_check.append({
            "h": h_label,
            "delta_EI": target,
            "deltaIE_peak (I_t)": die_peak,
            "Imax": Imax,
            "deltaIE (|w| min)": die_wzero,
            "w al minimo (NON zero esatto)": w_al_minimo,
            "differenza posizione": die_peak - die_wzero,
            "coincidono (tol=1 passo griglia)": match,
        })

T_check = pd.DataFrame(rows_check)
display(T_check)

n_match = T_check["coincidono (tol=1 passo griglia)"].sum()
n_tot = len(T_check)
print(f"\n{n_match}/{n_tot} casi (h x delta_EI) hanno il picco di I_t esattamente")
print(f"nello stesso punto di griglia (o al piu' un passo di distanza, {grid_step:.2f}) dello zero di w.")
print(f"\nPasso della griglia in delta_IE: {grid_step:.3f}")
print("Nota: 'w al minimo' non e' mai esattamente 0 - e' il valore di w nel punto di")
print("griglia piu' vicino al vero zero continuo, che in generale cade tra due punti")
print("di griglia adiacenti. E' 'piccolo' (ordini di grandezza sotto ai valori tipici")
print("di w altrove sulla stessa linea), non esattamente nullo.")

if n_match < n_tot:
    print("\nATTENZIONE: non tutti i casi coincidono - controllare le righe con")
    print("'coincidono'=False sopra.")

,h,delta_EI,deltaIE_peak (I_t),Imax,deltaIE (|w| min),w al minimo (NON zero esatto),differenza posizione,coincidono (tol=1 passo griglia)
0,10^{-6},2.5,-2.2,0.000015,-2.2,0.239975,0.0,True
1,10^{-6},5.0,-3.3,0.000010,-3.3,-0.311463,0.0,True
2,10^{-7},2.5,-2.2,0.000015,-2.2,0.239973,0.0,True
3,10^{-7},5.0,-3.3,0.000010,-3.3,-0.311467,0.0,True
4,10^{-8},2.5,-2.2,0.000015,-2.2,0.239972,0.0,True
5,10^{-8},5.0,-3.3,0.000010,-3.3,-0.311467,0.0,True
6,10^{-9},2.5,-2.2,0.000015,-2.2,0.239972,0.0,True
7,10^{-9},5.0,-3.3,0.000010,-3.3,-0.311467,0.0,True



8/8 casi (h x delta_EI) hanno il picco di I_t esattamente
nello stesso punto di griglia (o al piu' un passo di distanza, 0.10) dello zero di w.

Passo della griglia in delta_IE: 0.100
Nota: 'w al minimo' non e' mai esattamente 0 - e' il valore di w nel punto di
griglia piu' vicino al vero zero continuo, che in generale cade tra due punti
di griglia adiacenti. E' 'piccolo' (ordini di grandezza sotto ai valori tipici
di w altrove sulla stessa linea), non esattamente nullo.
